In [ ]:
import torch
from transformer_lens import HookedTransformer
from sae_lens import SAE
from sae_lens.analysis.neuronpedia_integration import get_neuronpedia_feature

# ============================================================
# CONFIG — change PRESET_NAME (or LAYER) to swap model / layer / SAE width.
# Everything else in the notebook derives from these, so this
# should be the only place you need to edit when experimenting
# (this is also what will become the CLI args once this becomes a script).
#
# HOOK_TEMPLATE is NOT the model's internal hook point name in general - it's
# the template for the `sae_id` string that SAE.from_pretrained() uses to find
# the right file within that release's repo. For gpt2-small-res-jb it happens to
# match TransformerLens's real hook name, but other releases use unrelated naming
# (path-like, or terse codes) - the actual hook point is read back afterward from
# sae.cfg.metadata.hook_name, so nothing downstream needs to know which scheme is used.
#
# Verified against SAELens' own pretrained_saes.yaml (not from memory) - Gemma 3 SAEs
# do exist, released as "gemma-scope-2-*" (the "2" = 2nd-gen GemmaScope suite, applied
# to Gemma 3 - a confusing name, NOT "for Gemma 2"). Unlike gpt2-small/Gemma 2 "canonical"
# releases (one SAE per layer, every layer), Gemma 3 SAEs only exist at a handful of
# layers per model size - LAYER must be one of the values noted below, not any integer.
# ============================================================
PRESETS = {
    "gpt2-small":   dict(model="gpt2-small",              release="gpt2-small-res-jb",                hook_template="blocks.{layer}.hook_resid_pre",     valid_layers="any of 0-11"),
    "gemma-1-2b":   dict(model="gemma-2b",                release="gemma-2b-res-jb",                  hook_template="blocks.{layer}.hook_resid_post",    valid_layers="any of 0-17"),
    "gemma-2-2b":   dict(model="gemma-2-2b",              release="gemma-scope-2b-pt-res-canonical",  hook_template="layer_{layer}/width_16k/canonical", valid_layers="any of 0-25"),
    "gemma-2-9b":   dict(model="gemma-2-9b",              release="gemma-scope-9b-pt-res-canonical",  hook_template="layer_{layer}/width_16k/canonical", valid_layers="any of 0-41 (model used in the CuE paper)"),
    "gemma-3-1b":   dict(model="google/gemma-3-1b-pt",    release="gemma-scope-2-1b-pt-res",          hook_template="layer_{layer}_width_16k_l0_medium", valid_layers="13, 17, or 22 only"),
    "gemma-3-4b":   dict(model="google/gemma-3-4b-pt",    release="gemma-scope-2-4b-pt-res",          hook_template="layer_{layer}_width_16k_l0_medium", valid_layers="9, 17, 22, or 29 only"),
    "gemma-3-12b":  dict(model="google/gemma-3-12b-pt",   release="gemma-scope-2-12b-pt-res",         hook_template="layer_{layer}_width_16k_l0_medium", valid_layers="24, 31, or 41 only"),
    "gemma-3-27b":  dict(model="google/gemma-3-27b-pt",   release="gemma-scope-2-27b-pt-res",         hook_template="layer_{layer}_width_16k_l0_medium", valid_layers="31, 40, or 53 only"),
    "llama-3.1-8b": dict(model="meta-llama/Llama-3.1-8B", release="llama_scope_lxr_32x",              hook_template="l{layer}r_32x",                     valid_layers="any of 0-31 (note: no width/sparsity choice in this release)"),
}

PRESET_NAME = "gpt2-small"                  # <-- pick one of the keys in PRESETS above
LAYER = 6                                    # <-- which layer to probe/steer; must be valid for the chosen preset (see "valid_layers" above)

_preset = PRESETS[PRESET_NAME]               # look up the rest of the config from the chosen preset
MODEL_NAME = _preset["model"]                # transformer_lens/HF model id
SAE_RELEASE = _preset["release"]             # SAE release name from sae_lens/pretrained_saes.yaml
HOOK_TEMPLATE = _preset["hook_template"]     # sae_id template for this release (see note above - not always a literal hook name)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")  # prefer GPU: MPS on Mac, CUDA elsewhere, else CPU
torch.set_grad_enabled(False)  # everything here is inference only - disable autograd globally to save memory/time

model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)  # load the base language model onto the chosen device
sae = SAE.from_pretrained(
    release=SAE_RELEASE,                              # which family of pretrained SAEs to pull from
    sae_id=HOOK_TEMPLATE.format(layer=LAYER),          # the specific layer/hook within that family, built from LAYER above
    device=device,                                     # keep the SAE on the same device as the model
)

def top_features(text, k=10):
    tokens = model.to_tokens(text)                # tokenize the text (BOS token prepended by default)
    token_strs = model.to_str_tokens(text)         # same tokens as human-readable strings, same order/length
    _, cache = model.run_with_cache(tokens, prepend_bos=True)  # run the model, keep every internal activation
    acts = sae.encode(cache[sae.cfg.metadata.hook_name])[0]    # decompose layer activations into SAE features: [seq, n_features]
    acts_no_bos = acts[1:]                         # drop position 0 (BOS) - its activation is content-independent and would dominate every result
    max_vals, argmax_pos = acts_no_bos.max(dim=0)  # per feature: its strongest activation in this text + which token position produced it
    top = torch.topk(max_vals, k)                  # the k features with the highest such activation

    out = []
    for feat_idx, val in zip(top.indices.tolist(), top.values.tolist()):
        pos = argmax_pos[feat_idx].item() + 1       # +1 undoes the BOS removal, back to the real token position
        out.append((feat_idx, val, pos, token_strs[pos]))  # (feature id, activation value, token position, token text)
    return out

# model_id/layer_str/dataset are read back FROM the SAE's own metadata (not from MODEL_NAME/LAYER above),
# so this stays correct automatically no matter which preset/layer is chosen in the config block.
# NOTE: this assumes the release has a Neuronpedia mapping in the "model/layer-dataset" format gpt2-small-res-jb
# uses - not every release/layer is guaranteed to have one (e.g. thin Gemma 3 releases might not), in which case
# explain() below will just fail to look anything up and you can skip/stub it out.
model_id, source_id = sae.cfg.metadata.neuronpedia_id.split("/")  # e.g. "gpt2-small/6-res-jb" -> ("gpt2-small", "6-res-jb")
layer_str, dataset = source_id.split("-", 1)                       # "6-res-jb" -> ("6", "res-jb")

_explain_cache = {}   # avoid re-fetching the same feature's explanation more than once
def explain(feature_idx):
    if feature_idx not in _explain_cache:
        data = get_neuronpedia_feature(feature=feature_idx, layer=int(layer_str), model=model_id, dataset=dataset)  # look up feature on Neuronpedia
        _explain_cache[feature_idx] = [e["description"] for e in data["explanations"]]  # keep just the human-readable descriptions
    return _explain_cache[feature_idx]


In [5]:
import pandas as pd
import re
from collections import Counter, defaultdict

df = pd.read_csv("WorldView-Bench Dataset.csv")   # one column per category, one row per question

def parse_question(raw):
    m = re.match(r"(\d+)\.\s*(.*)", raw)           # split "12. Some question?" into (12, "Some question?")
    return (int(m.group(1)), m.group(2)) if m else (None, raw)

results = {}   # category -> {"by_count": [...], "by_max": [...]}

for category in df.columns:                                            # process one category (column) at a time
    questions = [parse_question(raw) for raw in df[category].dropna()]  # [(question_id, question_text), ...]

    counts = Counter()                 # feature_idx -> in how many questions it appeared
    max_value = defaultdict(float)     # feature_idx -> strongest activation seen across the category
    max_source = {}                    # feature_idx -> (question_id, token) that produced that max
    question_ids = defaultdict(list)   # feature_idx -> list of question ids where it showed up

    for qid, q in questions:
        for idx, val, pos, token in top_features(q, k=10):   # top-10 candidate features for this one question
            counts[idx] += 1
            question_ids[idx].append(qid)
            if val > max_value[idx]:            # keep only the single strongest occurrence per feature
                max_value[idx] = val
                max_source[idx] = (qid, token)

    by_count = counts.most_common(10)                                            # 10 features appearing in the most questions
    by_max = sorted(max_value.items(), key=lambda kv: kv[1], reverse=True)[:10]  # 10 features with the highest single activation

    results[category] = {
        "by_count": [(idx, count, max_value[idx], question_ids[idx]) for idx, count in by_count],
        "by_max": [(idx, max_val, counts[idx], max_source[idx]) for idx, max_val in by_max],
    }
    print(f"processed {category} ({len(questions)} questions)")


processed Ethical / Moral (25 questions)
processed Religious (25 questions)
processed Lifestyle (25 questions)
processed Cultural Norms (25 questions)
processed Traditions (25 questions)
processed History (25 questions)
processed Technology (25 questions)


In [46]:
for category, data in results.items():
    print(f"\n=== {category} ===")

    print("-- top 10 by frequency (# of questions it showed up in) --")
    for idx, count, max_val, qids in data["by_count"]:          # qids = which question numbers triggered this feature
        descs = "; ".join(explain(idx)) or "(no explanation on Neuronpedia)"
        print(f"  feature {idx:>6}  count={count:2d}/25  max_act={max_val:8.2f}  Q{qids}  — {descs}")

    print("-- top 10 by max activation --")
    for idx, max_val, count, (qid, token) in data["by_max"]:    # (qid, token) = question + word that produced the single strongest hit
        descs = "; ".join(explain(idx)) or "(no explanation on Neuronpedia)"
        print(f"  feature {idx:>6}  max_act={max_val:8.2f}  count={count:2d}/25  Q{qid}={token!r}  — {descs}")



=== Ethical / Moral ===
-- top 10 by frequency (# of questions it showed up in) --
  feature  22431  count=16/25  max_act=   29.18  Q[1, 2, 3, 5, 6, 8, 9, 10, 12, 14, 15, 17, 19, 21, 24, 25]  — questions within sentences;  rhetorical questions
  feature  19151  count=10/25  max_act=   23.12  Q[2, 5, 7, 9, 10, 12, 18, 22, 24, 25]  — specific entities, possibly related to legal documents or paperwork; end-of-text tokens
  feature  20278  count= 9/25  max_act=   40.37  Q[2, 4, 6, 9, 12, 13, 18, 20, 23]  — phrases starting with "Should."; questions or propositions that begin with "Should."
  feature   1805  count= 8/25  max_act=   57.28  Q[2, 3, 7, 8, 11, 19, 24, 25]  — words related to morality; references to moral and ethical concepts
  feature  11672  count= 6/25  max_act=   30.30  Q[1, 5, 8, 11, 14, 21]  — the pronoun "it" used in various statements;  repeated uses of the word "it" in various contexts
  feature  23940  count= 6/25  max_act=   30.72  Q[1, 5, 8, 11, 14, 21]  — phrases i

## Reproducing Cultural Embeddings (CuE) — Khanuja et al., *Steering LLMs for Culturally Localized Generation*

Adapts the paper's Steps 1-2-3b (feature extraction → mutual-information feature selection → steering vector) using the real **CANDLE** dataset (Nguyen et al., 2023), restricted to the `rituals` and `traditions` facets only.

Model/layer are now single config variables at the top of the setup cell (`MODEL_NAME`, `SAE_RELEASE`, `LAYER`, `HOOK_TEMPLATE`) — change those three to try a different model or layer; everything else in the notebook derives from them.

**Religion pipeline commented out for now**: CANDLE labels religion assertions by religion (Islam, Christianity, ...) and country assertions by country — there's no single assertion carrying both labels, so religion prototypes can't be cross-referenced against country prototypes the way we'd hoped. Left commented out rather than deleted, in case a country+religion-labeled data source turns up.

Simplifications vs. the paper (flagged so results are read with the right caveats):
- Single SAE layer instead of steering across every layer of a much larger model.
- Country names are stripped with a regex instead of the paper's LLM-based rewrite + GPT-4.1 augmentation (Appendix A) — some lexical leakage may remain, which the paper shows inflates early-layer feature selection.
- Mutual information uses a binary "did this feature fire at all" discretization rather than a learned/binned estimator — a common simplification for sparse SAE activations, but a design choice worth revisiting if results look off.


In [ ]:
import json     # for parsing the CANDLE JSONL files
import random   # for reproducible random sampling of assertions per group

random.seed(0)  # fixed seed so sampled assertions are identical across reruns

def load_candle_groups(path, label_key):
    # Reads a CANDLE JSONL subset file and groups assertion text by a chosen label column.
    groups = {}                                 # label -> list of assertion strings
    with open(path) as f:                       # open the JSONL file
        for line in f:                          # one JSON object per line
            row = json.loads(line)              # parse the line into a dict
            groups.setdefault(row[label_key], []).append(row["assertion"])  # bucket by label (e.g. "country")
    return groups

def sample_groups(groups, n_per_group=100):
    # Uniformly samples up to n_per_group assertions per label, for balanced per-group data (paper Sec 2).
    return {g: random.sample(texts, min(n_per_group, len(texts))) for g, texts in groups.items()}

# Countries: CANDLE domain="countries", facet in {rituals, traditions} only, the 22 countries with
# >=500 total assertions (paper's Sec 2 selection).
country_groups = load_candle_groups("candle_countries_subset.jsonl", "country")   # label_key="country" column in the JSONL
countries = sorted(country_groups.keys())                                          # alphabetical list of the 22 country labels
sampled_country = sample_groups(country_groups, n_per_group=100)                   # up to 100 assertions per country
print(f"{len(countries)} countries:", countries)
for c in countries:
    print(f"  {c}: {len(sampled_country[c])} assertions sampled (of {len(country_groups[c])} available)")

# --- Religion pipeline disabled for now ---
# CANDLE's "religions" domain labels assertions by religion (Islam, Christianity, ...), NOT by country -
# there's no assertion in CANDLE carrying both a country AND a religion label at once, so this can't be
# cross-referenced against the country prototypes below. Commented out (not deleted) until we have a
# data source that actually pairs religion with country, or we decide to treat it as its own label set again.
# religion_groups = load_candle_groups("candle_religions_subset.jsonl", "religion")
# religions = sorted(religion_groups.keys())
# sampled_religion = sample_groups(religion_groups, n_per_group=100)
# print(f"\n{len(religions)} religions:", religions)
# for r in religions:
#     print(f"  {r}: {len(sampled_religion[r])} assertions sampled (of {len(religion_groups[r])} available)")


In [ ]:
import re  # for building the name-stripping regex patterns

# Paper removes explicit label names via an LLM rewrite before feature discovery (Sec 2.1, App. A) so
# features don't just learn "detect the literal name token." We approximate with a regex strip of each
# label's name + common adjective/demonym forms - cruder than an LLM rewrite, but dependency-free.
COUNTRY_ALIASES = {                       # country label -> surface forms to strip out of its own assertions
    "Australia": ["Australia", "Australian"],
    "Canada": ["Canada", "Canadian"],
    "China": ["China", "Chinese"],
    "Egypt": ["Egypt", "Egyptian"],
    "England": ["England", "English"],
    "France": ["France", "French"],
    "Germany": ["Germany", "German"],
    "Greece": ["Greece", "Greek"],
    "India": ["India", "Indian"],
    "Ireland": ["Ireland", "Irish"],
    "Israel": ["Israel", "Israeli"],
    "Italy": ["Italy", "Italian"],
    "Japan": ["Japan", "Japanese"],
    "Mexico": ["Mexico", "Mexican"],
    "Russia": ["Russia", "Russian"],
    "Scotland": ["Scotland", "Scottish", "Scots"],
    "South Korea": ["South Korea", "Korean", "Korea"],
    "Spain": ["Spain", "Spanish", "Spaniard"],
    "Thailand": ["Thailand", "Thai"],
    "Turkey": ["Turkey", "Turkish", "Turk"],
    "United Kingdom": ["United Kingdom", "UK", "British", "Britain"],
    "United States": ["United States", "USA", "U.S.", "US", "American"],
}

# --- Religion alias map disabled along with the religion pipeline above ---
# RELIGION_ALIASES = {
#     "Islam": ["Islam", "Islamic", "Muslim"],
#     "Christianity": ["Christianity", "Christian"],
#     "Hinduism": ["Hinduism", "Hindu"],
#     "Buddhism": ["Buddhism", "Buddhist"],
#     "Catholicism": ["Catholicism", "Catholic"],
#     "Jews": ["Jews", "Jewish", "Judaism", "Jew"],
#     "Sikhism": ["Sikhism", "Sikh"],
#     "Shinto": ["Shinto", "Shintoism", "Shintoist"],
#     "Protestantism": ["Protestantism", "Protestant"],
#     "Shia Islam": ["Shia Islam", "Shia", "Shiite", "Shias"],
#     "Vodou": ["Vodou", "Voodoo"],
# }

def make_stripper(alias_map):
    # Builds one regex per label from its alias list, matching whole words case-insensitively.
    patterns = {
        label: re.compile("|".join(rf"\b{re.escape(a)}\w*\b" for a in aliases), re.IGNORECASE)   # \w* also catches plurals/suffixes
        for label, aliases in alias_map.items()
    }
    def strip(text, label):
        text = patterns[label].sub("", text)          # remove every match of this label's own name/aliases
        return re.sub(r"\s+", " ", text).strip()       # collapse any doubled-up spaces left behind, trim ends
    return strip

strip_country = make_stripper(COUNTRY_ALIASES)   # ready-to-use stripper function for country assertions
# strip_religion = make_stripper(RELIGION_ALIASES)  # disabled along with the religion pipeline

def build_dataset(sampled, groups, strip_fn):
    # Flattens the {label: [texts]} dict into parallel (assertions, labels) lists, stripping each text's own label name.
    texts, labels = [], []
    for g in groups:                        # iterate labels in a fixed order (alphabetical, from `countries`/`religions`)
        for text in sampled[g]:              # every sampled assertion for this label
            texts.append(strip_fn(text, g))   # remove this group's own name so the model can't just pattern-match it
            labels.append(g)                  # parallel label array, same length/order as texts
    return texts, labels

assertions_country, labels_country = build_dataset(sampled_country, countries, strip_country)
# assertions_religion, labels_religion = build_dataset(sampled_religion, religions, strip_religion)  # disabled

print(f"{len(assertions_country)} country assertions")
print("country example:", assertions_country[0], "->", labels_country[0])


In [ ]:
import numpy as np           # for building/storing the activation matrix
from tqdm.auto import tqdm   # progress bar - this loop runs one forward pass per assertion, the slow step

def get_full_activation(text):
    # Returns the FULL max-pooled SAE feature vector for one text (all n_features dims, not just top-k).
    tokens = model.to_tokens(text)                              # tokenize (BOS prepended by default)
    _, cache = model.run_with_cache(tokens, prepend_bos=True)    # run the model, keep every internal activation
    acts = sae.encode(cache[sae.cfg.metadata.hook_name])[0]      # decompose into SAE features: [seq, n_features]
    return acts[1:].max(dim=0).values                            # max-pool over tokens, skip BOS - this is a(x) from Sec 2.2

def extract_activation_matrix(assertions):
    # Paper's Step 1: build one row of a(x) per assertion, stacked into an [n_assertions, n_features] matrix.
    X = None                                        # allocated once we know n_features, from the first assertion
    for i, text in enumerate(tqdm(assertions)):      # one forward pass per assertion
        vec = get_full_activation(text).cpu().numpy()   # move off GPU/MPS into a plain numpy array
        if X is None:
            X = np.zeros((len(assertions), vec.shape[0]), dtype=np.float32)  # allocate now that n_features is known
        X[i] = vec                                    # store this assertion's activation vector as row i
    return X

print("extracting country activations...")
X_country = extract_activation_matrix(assertions_country)   # [n_country_assertions, n_features]
y_country = np.array(labels_country)                          # parallel array of country labels
print("X_country:", X_country.shape, " (n_assertions x n_features)")

# print("extracting religion activations...")   # disabled along with the religion pipeline
# X_religion = extract_activation_matrix(assertions_religion)
# y_religion = np.array(labels_religion)
# print("X_religion:", X_religion.shape, " (n_assertions x n_features)")


In [ ]:
def compute_mi(X, y, groups):
    # Paper's Step 2 (Sec 2.3): I(A_j; C) = sum_{a_j,c} P(a_j,c) log( P(a_j,c) / (P(a_j)P(c)) )
    # We discretize each feature's activation into "fired" (>0) vs "silent" (==0) to get P(a_j, c) empirically.
    active = (X > 0)                                          # binary matrix: did feature j fire on assertion i?
    label_idx = {g: i for i, g in enumerate(groups)}           # map each label string to an integer index
    y_idx = np.array([label_idx[l] for l in y])                # convert the label array to integer indices
    n_groups = len(groups)
    p_g = np.array([(y_idx == i).mean() for i in range(n_groups)])   # P(c): fraction of assertions belonging to each group

    mi = np.zeros(active.shape[1], dtype=np.float64)   # will hold one MI value per SAE feature
    eps = 1e-12                                         # tiny constant to avoid log(0) / divide-by-zero
    p_active = active.mean(axis=0)          # P(a_j = 1): overall firing rate of each feature, across all assertions
    p_inactive = 1 - p_active               # P(a_j = 0): overall silent rate of each feature

    for i in range(n_groups):                            # accumulate MI contribution from each group/label in turn
        mask = (y_idx == i)                                # which assertions belong to this group
        p_active_given_g = active[mask].mean(axis=0)       # P(a_j=1 | c): firing rate of each feature within this group only
        joint1 = p_active_given_g * p_g[i]                  # P(a_j=1, c) via P(a_j=1|c) * P(c)
        mi += joint1 * np.log((joint1 + eps) / (p_active * p_g[i] + eps))   # "fired" term of the MI sum

        p_inactive_given_g = 1 - p_active_given_g           # P(a_j=0 | c)
        joint0 = p_inactive_given_g * p_g[i]                # P(a_j=0, c)
        mi += joint0 * np.log((joint0 + eps) / (p_inactive * p_g[i] + eps))  # "silent" term of the MI sum
    return mi

def select_top_mi(mi, rho=0.1):
    # Rank all features globally by MI, keep the top ones until cumulative MI reaches rho * total MI (App. C default).
    order = np.argsort(-mi)               # feature indices sorted by MI, highest first
    cum = np.cumsum(mi[order])            # running total of MI as features are added in that order
    cutoff = int(np.searchsorted(cum, rho * cum[-1])) + 1   # how many features are needed to reach rho fraction of total MI
    return order[:cutoff], order          # (selected feature set S, full ranking - the ranking is handy for inspection)

mi_country = compute_mi(X_country, y_country, countries)          # one MI value per feature, for the country label set
S_country, order_country = select_top_mi(mi_country)               # top-MI feature indices (paper's S) + full ranking
print(f"countries: selected |S| = {len(S_country)} / {len(mi_country)} features ({len(S_country) / len(mi_country) * 100:.2f}%)")

# mi_religion = compute_mi(X_religion, y_religion, religions)       # disabled along with the religion pipeline
# S_religion, order_religion = select_top_mi(mi_religion)
# print(f"religions: selected |S| = {len(S_religion)} / {len(mi_religion)} features ({len(S_religion) / len(mi_religion) * 100:.2f}%)")


In [ ]:
def build_prototypes(X, y, groups, S):
    # Sec 2.3 continued: CuE(x) = a(x)[S] (restrict activations to the selected feature set),
    # then average CuE(x) over all assertions of each group to get that group's prototype p_CuE^(c).
    CuE = X[:, S]                                    # keep only the culturally-informative columns
    return {g: CuE[y == g].mean(axis=0) for g in groups}   # one averaged vector per group/label

prototypes_country = build_prototypes(X_country, y_country, countries, S_country)   # {country: prototype vector}
print("country prototypes:", len(prototypes_country))

# prototypes_religion = build_prototypes(X_religion, y_religion, religions, S_religion)  # disabled along with the religion pipeline
# print("religion prototypes:", len(prototypes_religion))


In [ ]:
# Sanity check (paper's RQ2-style analysis): what do the highest-MI features actually mean?
print("Top 15 culturally-informative features by MI (COUNTRIES):")
for idx in order_country[:15]:                                 # walk the top 15 features by MI, highest first
    descs = "; ".join(explain(int(idx))) or "(no explanation on Neuronpedia)"   # human-readable Neuronpedia description(s)
    print(f"  feature {int(idx):>6}  MI={mi_country[idx]:.5f}  — {descs}")

# print("\nTop 15 culturally-informative features by MI (RELIGIONS):")   # disabled along with the religion pipeline
# for idx in order_religion[:15]:
#     descs = "; ".join(explain(int(idx))) or "(no explanation on Neuronpedia)"
#     print(f"  feature {int(idx):>6}  MI={mi_religion[idx]:.5f}  — {descs}")


In [ ]:
def build_steering_vector(target_label, groups, X, S, prototypes):
    # Sec 2.5: delta = p_CuE^(target) - mean(p_CuE^(other groups)), restricted to S.
    others = [g for g in groups if g != target_label]           # every group except the one we're steering toward
    p_target = prototypes[target_label]                          # the target group's prototype (in the |S|-dim CuE space)
    p_others_mean = np.mean([prototypes[g] for g in others], axis=0)   # average prototype of all OTHER groups
    delta = p_target - p_others_mean                              # direction distinctive to the target, relative to the rest

    delta_full = np.zeros(X.shape[1], dtype=np.float32)   # zero-pad delta back out to the full SAE feature space
    delta_full[S] = delta                                   # place delta's values only at the originally-selected feature indices

    delta_full_t = torch.tensor(delta_full, device=device, dtype=sae.W_dec.dtype)  # move to torch, matching the SAE decoder's dtype/device
    return delta_full_t @ sae.W_dec                        # decode feature-space direction into residual-stream space: v = W_dec^T @ delta


def hooked_generate(prompt, v_cue, alpha=1.0, max_new_tokens=40, temperature=0.9, seed=0):
    # Generates text with an optional steering vector added into the residual stream at every position.
    torch.manual_seed(seed)                     # fixed seed so unsteered/steered outputs are directly comparable
    tokens = model.to_tokens(prompt)             # tokenize the prompt

    def steering_hook(resid, hook):
        resid[:, :, :] += alpha * v_cue          # add alpha * steering vector to every token's residual stream activation
        return resid

    fwd_hooks = [(sae.cfg.metadata.hook_name, steering_hook)] if alpha != 0 else []   # skip the hook entirely for alpha=0 (unsteered baseline)
    with model.hooks(fwd_hooks=fwd_hooks):        # register the hook only for the duration of this generate() call
        out = model.generate(
            input=tokens,
            max_new_tokens=max_new_tokens,        # how many new tokens to sample
            do_sample=True,                        # sample rather than greedy-decode, for more natural text
            temperature=temperature,                # sampling temperature
            stop_at_eos=False,                      # avoids a known bug on MPS where generation can hang/crash on EOS
        )
    return model.to_string(out[0])                  # decode the generated token ids back into text


prompt = "Write me a recipe for a local dish."   # a culture-agnostic prompt, like the paper's evaluation set

v_country = build_steering_vector("Japan", countries, X_country, S_country, prototypes_country)  # steering vector toward Japan
print("UNSTEERED:\n", hooked_generate(prompt, v_country, alpha=0.0))                                # baseline, no steering applied
print("\nSTEERED toward Japan (country CuE, alpha=4):\n", hooked_generate(prompt, v_country, alpha=4.0))  # alpha=4 is a starting guess - tune empirically

# v_religion = build_steering_vector("Buddhism", religions, X_religion, S_religion, prototypes_religion)  # disabled along with the religion pipeline
# print("\nSTEERED toward Buddhism (religion CuE, alpha=4):\n", hooked_generate(prompt, v_religion, alpha=4.0))
